## Exploratory data anaylsis of CyCIF dataset


In [1]:
import os
import sys
import gc
import cv2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import tifffile
import xml.etree.ElementTree as ET

from skimage.transform import rescale
from IPython.display import display

In [2]:
import matplotlib.pyplot as plt
import matplotlib.font_manager
from matplotlib import rcParams

rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

In [3]:
sys.path.append('..')
import IO, zonation

### CyCIF multiplexed image registration

#### Full 3D sample

| Cycle | Opal 520 | Opal 690 | Opal 570 | Opal 780 | 
| --- | --- | --- | --- | --- |
| 1 | B-Catenin | ASS1 | GS | CYP3A4 |
| 2 | Pan-CK | CD31 | Collagen | NaN |
| 3 | CD45 | CD68 | Arg1 | Lyve1 |
| 4 | CD56 | Vimentin | PU1 | CD3 |

(1). Load Cy-IF images from google cloud bucket

In [4]:
CREDENTIAL_PATH = "../data/azizilab-cell-segmentation-05f1a1125db2.json"
PROJECT_ID = 'azizilab-cell-segmentation'
BUCKET_ID = 'liver_3d'
HOME_PATH = 'CyIF/qupath/'

cyif_reader = IO.CyIFGcloudReader(CREDENTIAL_PATH,
                                  BUCKET_ID,
                                  PROJECT_ID,
                                  HOME_PATH,
                                  scale=1,
                                  sigma=10)


(2). Within-cycle registration

In [5]:
out_path = '../data/cycif/Cy-IF_downsample'

for slide_id in cyif_reader.slide_ids:
    # Redo slide #6 to correct the DAPI registration
    if slide_id != 'CyIF-6':
        continue

    annot_imgs = cyif_reader.load_imgs(slide_id,
                                       chans_to_ignore={'Opal 620'},
                                       verbose=True)    
    
    annot_imgs_warped = cyif_reader.register_cycles(annot_imgs, 
                                                    verbose=True)
    
    cyif_reader.save_annotated_tiff(annot_imgs_warped, 
                                    output_path=out_path)

    gc.collect()
    # del annot_imgs, annot_imgs_warped

[2024-01-11 00:19:56] Loading images from Slide CyIF-6...
[2024-01-11 00:20:04] 	Loading CyIF_06_01_00.qptiff...
[2024-01-11 00:20:30] 	Loading CyIF_06_01_01.qptiff...
[2024-01-11 00:20:53] 	Loading CyIF_06_01_02.qptiff...
[2024-01-11 00:21:16] 	Loading CyIF_06_01_03.qptiff...
[2024-01-11 00:21:39] 	Loading CyIF_06_02_00.qptiff...
[2024-01-11 00:21:59] 	Loading CyIF_06_02_01.qptiff...
[2024-01-11 00:22:19] 	Loading CyIF_06_02_02.qptiff...
[2024-01-11 00:22:39] 	Loading CyIF_06_02_03.qptiff...
[2024-01-11 00:22:59] 	Loading CyIF_06_03_00.qptiff...
[2024-01-11 00:23:21] 	Loading CyIF_06_03_01.qptiff...
[2024-01-11 00:23:44] 	Loading CyIF_06_03_02.qptiff...
[2024-01-11 00:24:04] 	Loading CyIF_06_03_03.qptiff...
[2024-01-11 00:24:27] 	Loading CyIF_06_04_00.qptiff...
[2024-01-11 00:24:52] 	Loading CyIF_06_04_01.qptiff...
[2024-01-11 00:25:17] 	Loading CyIF_06_04_02.qptiff...
[2024-01-11 00:25:42] 	Loading CyIF_06_04_03.qptiff...
[2024-01-11 00:26:07] Registering channels from Slide CyIF_06,

Re-calculated pts w/ larger search space


[2024-01-11 00:28:47] Registering channels from Slide CyIF_06, Tissue 02...


Re-calculated pts w/ larger search space


[2024-01-11 00:30:03] Registering channels from Slide CyIF_06, Tissue 03...


Re-calculated pts w/ larger search space


[2024-01-11 00:31:20] Saving 19-chan image CyIF_tiss_20.ome.tif...
[2024-01-11 00:31:21] Saving 20-chan image CyIF_tiss_21.ome.tif...
[2024-01-11 00:31:22] Saving 19-chan image CyIF_tiss_22.ome.tif...
[2024-01-11 00:31:22] Saving 20-chan image CyIF_tiss_23.ome.tif...


### Intensity Adjustment

Try both heuristic & data-driven

In [4]:
from skimage.exposure import adjust_gamma

(1). Heuristic min-max intensity & gamma adjustment

In [5]:
data_path = '../data/cycif/Cy-IF_downsample'
out_path = '../data/cycif/Cy-IF_downsample_adj'

if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

In [7]:
# TODO: add util function for local `qptiff` & `ome.tiff` IO

CREDENTIAL_PATH = "../data/azizilab-cell-segmentation-05f1a1125db2.json"
PROJECT_ID = 'azizilab-cell-segmentation'
BUCKET_ID = 'liver_3d'
HOME_PATH = 'CyIF/qupath/'

cyif_reader = IO.CyIFGcloudReader(CREDENTIAL_PATH,
                                  BUCKET_ID,
                                  PROJECT_ID,
                                  HOME_PATH)

filenames = sorted([os.path.join(data_path, f)
                    for f in os.listdir(data_path)
                    if f[-7:] == 'ome.tif'])

chan_meta_df = pd.read_csv(os.path.join(data_path, 'CyIF_antibody_ref_sheet.csv'), index_col=[0])
display(chan_meta_df.head())

int_meta_df = pd.read_csv(os.path.join(data_path, 'CyIF_intensity_ref_sheet.csv'), index_col=[0, 1, 2, 3])
int_meta_df.dropna(axis=1, how='all', inplace=True)
int_meta_df = int_meta_df.applymap(lambda x: x if pd.isna(x) else min(x, 255.0))

display(int_meta_df.head())

,Opal_520,Opal_570,Opal_690,Opal_780
Round Number,,,,
1,B-catenin-AF 488,ASS1 PE,GS 647,CYP3A4
2,Pan CK,CD31,Col I,NaN
3,CD45,CD68,Arg1,Lyve1
4,CD56,Vimentin,PU1,CD3


Min-Display_DAPI  \
Slide Number Tissue Round Number File Name                         
1            1      0            CyIF_01_00_00               0.0   
                    1            CyIF_01_01_00               0.0   
                    2            CyIF_01_02_00               0.0   
                    3            CyIF_01_03_00               0.0   
                    4            CyIF_01_04_00               0.0   

                                                Max-Display_DAPI  \
Slide Number Tissue Round Number File Name                         
1            1      0            CyIF_01_00_00             255.0   
                    1            CyIF_01_01_00              67.7   
                    2            CyIF_01_02_00              85.4   
                    3            CyIF_01_03_00             255.0   
                    4            CyIF_01_04_00             255.0   

                                                Min-Display_Opal_520  \
Slide Number Tissue Round Number File Name                             
1            1      0            CyIF_01_00_00                  0.00   
                    1            CyIF_01_01_00                  0.00   
                    2            CyIF_01_02_00                 23.60   
                    3            CyIF_01_03_00                  7.73   
                    4            CyIF_01_04_00                 41.00   

                                                Max-Display_Opal_520  \
Slide Number Tissue Round Number File Name                             
1            1      0            CyIF_01_00_00                 255.0   
                    1            CyIF_01_01_00                 255.0   
                    2            CyIF_01_02_00                 149.7   
                    3            CyIF_01_03_00                 123.7   
                    4            CyIF_01_04_00                 255.0   

                                                Min-Display_Opal_570  \
Slide Number Tissue Round Number File Name                             
1            1      0            CyIF_01_00_00                  0.00   
                    1            CyIF_01_01_00                 29.00   
                    2            CyIF_01_02_00                  9.05   
                    3            CyIF_01_03_00                 28.10   
                    4            CyIF_01_04_00                 47.20   

                                                Max-Display_Opal_570  \
Slide Number Tissue Round Number File Name                             
1            1      0            CyIF_01_00_00                 255.0   
                    1            CyIF_01_01_00                 169.2   
                    2            CyIF_01_02_00                 117.0   
                    3            CyIF_01_03_00                 255.0   
                    4            CyIF_01_04_00                 255.0   

                                                Min-Display_Opal_690  \
Slide Number Tissue Round Number File Name                             
1            1      0            CyIF_01_00_00                  0.00   
                    1            CyIF_01_01_00                 67.40   
                    2            CyIF_01_02_00                 10.90   
                    3            CyIF_01_03_00                 18.80   
                    4            CyIF_01_04_00                  7.83   

                                                Max-Display_Opal_690  \
Slide Number Tissue Round Number File Name                             
1            1      0            CyIF_01_00_00                 255.0   
                    1            CyIF_01_01_00                 252.0   
                    2            CyIF_01_02_00                 141.0   
                    3            CyIF_01_03_00                  55.0   
                    4            CyIF_01_04_00                  41.5   

                                                Min-Display_Opal

In [13]:
# TMP: Switch channel orders starting from Slide 2
# swap (ASS, GS), (Col I, CD31), (Arg1, CD68), (Vimentin, PU1)

# tmp_filenames = filenames[4:]
# tmp_chan_lists = []

# for f in tmp_filenames:
#     ifile = open(f, 'rb')
#     tmp_chan_lists.append(IO.load_ome_labels(ifile, f)) 

# tissue_ids = [f.rpartition('/')[-1]
#               for f in tmp_filenames]

# # Swap orders (likely due to labeling issues btw Opal_570 & Opal_680)
# for i in range(len(tmp_chan_lists)):
#     idx1, idx2 = tmp_chan_lists[i].index('GS 647'), tmp_chan_lists[i].index('ASS1 PE')
#     tmp_chan_lists[i][idx1], tmp_chan_lists[i][idx2] = tmp_chan_lists[i][idx2], tmp_chan_lists[i][idx1]
    
#     idx1, idx2 = tmp_chan_lists[i].index('Col I'), tmp_chan_lists[i].index('CD31')
#     tmp_chan_lists[i][idx1], tmp_chan_lists[i][idx2] = tmp_chan_lists[i][idx2], tmp_chan_lists[i][idx1]

#     idx1, idx2 = tmp_chan_lists[i].index('Arg1'), tmp_chan_lists[i].index('CD68')
#     tmp_chan_lists[i][idx1], tmp_chan_lists[i][idx2] = tmp_chan_lists[i][idx2], tmp_chan_lists[i][idx1]

#     idx1, idx2 = tmp_chan_lists[i].index('PU1'), tmp_chan_lists[i].index('Vimentin')
#     tmp_chan_lists[i][idx1], tmp_chan_lists[i][idx2] = tmp_chan_lists[i][idx2], tmp_chan_lists[i][idx1]
    
# del i, idx1, idx2

# # Annotated imgs
# annot_imgs = {}
# for f, tid, labels in zip(tmp_filenames, tissue_ids, tmp_chan_lists):
#     img = tifffile.imread(f)
#     annot_img = {c: img_chan 
#                  for (c, img_chan) in zip(labels, img)}
    
#     annot_imgs[tid] = annot_img

# del f, tid, ifile, labels

# cyif_reader.save_annotated_tiff(annot_imgs, output_path=out_path, verbose=True)


[2024-01-11 01:02:15] Saving 20-chan image CyIF_tiss_04.ome.tif...
[2024-01-11 01:02:16] Saving 20-chan image CyIF_tiss_05.ome.tif...
[2024-01-11 01:02:17] Saving 20-chan image CyIF_tiss_06.ome.tif...
[2024-01-11 01:02:18] Saving 20-chan image CyIF_tiss_07.ome.tif...
[2024-01-11 01:02:20] Saving 20-chan image CyIF_tiss_08.ome.tif...
[2024-01-11 01:02:21] Saving 20-chan image CyIF_tiss_09.ome.tif...
[2024-01-11 01:02:22] Saving 20-chan image CyIF_tiss_10.ome.tif...
[2024-01-11 01:02:23] Saving 20-chan image CyIF_tiss_11.ome.tif...
[2024-01-11 01:02:24] Saving 20-chan image CyIF_tiss_12.ome.tif...
[2024-01-11 01:02:25] Saving 20-chan image CyIF_tiss_13.ome.tif...
[2024-01-11 01:02:26] Saving 20-chan image CyIF_tiss_14.ome.tif...
[2024-01-11 01:02:27] Saving 20-chan image CyIF_tiss_15.ome.tif...
[2024-01-11 01:02:28] Saving 20-chan image CyIF_tiss_16.ome.tif...
[2024-01-11 01:02:29] Saving 20-chan image CyIF_tiss_17.ome.tif...
[2024-01-11 01:02:30] Saving 20-chan image CyIF_tiss_18.ome.ti

In [8]:
def adj_intensity(fname, chan_meta, int_meta, verbose=True):
    if verbose:
        print('Adjusting intensities for img {}...'.format(
            fname.rpartition('/')[-1]))

    # Load annotated image from '.tif'
    img = tifffile.imread(fname)
    ifile = open(fname, 'rb')
    labels = IO.load_ome_labels(ifile, fname)
    annot_img = {lbl: img_chan
                 for (lbl, img_chan) in zip(labels, img)}
    
    adjusted_img = {}
    adjusted_img['DAPI'] = annot_img['DAPI']

    # Adjust intensities
    cycles = chan_meta.index
    channels = chan_meta.columns

    for cycle in cycles:
        for channel in channels:
            lbl = chan_meta.loc[cycle, channel]
            if pd.isna(lbl) or lbl not in annot_img.keys():
                continue

            if verbose:
                print('\tChannel: {}'.format(lbl))

            # Retrieve heursitic target intensities
            mask_row = int_meta.index.get_level_values('Round Number') == cycle
            mask_col = ['Min-Display_'+channel, 'Max-Display_'+channel, 'Gamma']
            imin, imax, gamma = int_meta.loc[mask_row, mask_col].values.flatten()

            X = annot_img[lbl] # intensity of the given image chan
            X = (X - imin) / (imax - imin)
            X[X < 0] = 0
            X[X > 1] = 1
            X = adjust_gamma(X, gamma)

            # Rescale to [0, 255]
            adjusted_img[lbl] = np.round(255*X).astype(np.uint8)

    # Append AF channels
    for lbl in labels:
        if 'AF' in lbl:
            adjusted_img[lbl] = annot_img[lbl]

    return adjusted_img


In [9]:
adjusted_imgs = {}

for filename in filenames:
    key = filename.rpartition('/')[-1]
    tissue_id = int(filenames[-1][-10:-8]) + 1
    int_meta = int_meta_df.loc[int_meta_df.index.get_level_values('Tissue') == tissue_id]
    adjusted_img = adj_intensity(filename, chan_meta_df, int_meta) 
    adjusted_imgs[key] = adjusted_img

    del adjusted_img
    gc.collect()

del filename, key, tissue_id

Adjusting intensities for img CyIF_tiss_00.ome.tif...
	Channel: B-catenin-AF 488
	Channel: ASS1 PE
	Channel: GS 647
	Channel: CYP3A4
	Channel: Pan CK
	Channel: CD31
	Channel: Col I
	Channel: CD45
	Channel: CD68
	Channel: Arg1
	Channel: Lyve1
	Channel: CD56
	Channel: Vimentin
	Channel: PU1
	Channel: CD3
Adjusting intensities for img CyIF_tiss_01.ome.tif...
	Channel: B-catenin-AF 488
	Channel: ASS1 PE
	Channel: GS 647
	Channel: CYP3A4
	Channel: Pan CK
	Channel: CD31
	Channel: Col I
	Channel: CD45
	Channel: CD68
	Channel: Arg1
	Channel: Lyve1
	Channel: CD56
	Channel: Vimentin
	Channel: PU1
	Channel: CD3
Adjusting intensities for img CyIF_tiss_02.ome.tif...
	Channel: B-catenin-AF 488
	Channel: ASS1 PE
	Channel: GS 647
	Channel: CYP3A4
	Channel: Pan CK
	Channel: CD31
	Channel: Col I
	Channel: CD45
	Channel: CD68
	Channel: Arg1
	Channel: Lyve1
	Channel: CD56
	Channel: Vimentin
	Channel: PU1
	Channel: CD3
Adjusting intensities for img CyIF_tiss_03.ome.tif...
	Channel: B-catenin-AF 488
	Channe

In [11]:
cyif_reader.save_annotated_tiff(adjusted_imgs, output_path=out_path)

[2024-01-11 01:29:15] Saving 20-chan image CyIF_tiss_00.ome.tif...
[2024-01-11 01:29:16] Saving 20-chan image CyIF_tiss_01.ome.tif...
[2024-01-11 01:29:17] Saving 20-chan image CyIF_tiss_02.ome.tif...
[2024-01-11 01:29:18] Saving 20-chan image CyIF_tiss_03.ome.tif...
[2024-01-11 01:29:19] Saving 20-chan image CyIF_tiss_04.ome.tif...
[2024-01-11 01:29:20] Saving 20-chan image CyIF_tiss_05.ome.tif...
[2024-01-11 01:29:21] Saving 20-chan image CyIF_tiss_06.ome.tif...
[2024-01-11 01:29:23] Saving 20-chan image CyIF_tiss_07.ome.tif...
[2024-01-11 01:29:24] Saving 20-chan image CyIF_tiss_08.ome.tif...
[2024-01-11 01:29:25] Saving 20-chan image CyIF_tiss_09.ome.tif...
[2024-01-11 01:29:26] Saving 20-chan image CyIF_tiss_10.ome.tif...
[2024-01-11 01:29:28] Saving 20-chan image CyIF_tiss_11.ome.tif...
[2024-01-11 01:29:28] Saving 20-chan image CyIF_tiss_12.ome.tif...
[2024-01-11 01:29:30] Saving 20-chan image CyIF_tiss_13.ome.tif...
[2024-01-11 01:29:31] Saving 20-chan image CyIF_tiss_14.ome.ti

### 2D/3D zonation

- TODO: **Figure out misalignment issues**, currently only works for Slide 1 & 3
- Selected zonation markers: `GS 647`, `CYP3A4`, `ASS1 PE` (Round 1), `Col I`, `Pan CK` (Round 2)
- Try construct 3D graphs

In [ ]:
from skimage.filters import gaussian, sobel, threshold_otsu, threshold_minimum
from skimage.exposure import rescale_intensity
from skimage.registration import optical_flow_ilk
from scipy import ndimage as ndi

In [ ]:
from importlib import reload
reload(IO)

In [ ]:
def remove_holes(roi, min_area):
    """
    Remove holes & FP lslands in binary ROI mask
    """
    roi_filtered = roi.copy().astype(np.uint8)
    roi_labeled, n_features = ndi.label(roi)
    
    for i in range(1, n_features+1):
        if (roi_labeled == i).sum() < min_area:
            roi_filtered[roi_labeled == i] = 0
            
    return ndi.binary_fill_holes(roi_filtered).astype(np.uint8)


def get_roi_mask(dapi, sigma=10, min_area=None):
    """
    Compute slice foreground / background mask
    """
    if min_area is None:
        # Defaulting min_area as 1/4 of the whole FOV area
        min_area = dapi.shape[0]//2*dapi.shape[1]//2
        
    dapi_smoothed = rescale_intensity(gaussian(dapi, sigma=10), out_range=(0, 1))
    threshold = threshold_otsu(dapi_smoothed)
    return remove_holes(dapi_smoothed > threshold, min_area=min_area)
    

Load aligned CyIF tiffs:

In [ ]:
data_path = '../data/cycif/Cy-IF_downsample_aln/registered_slides/'
cyif_imgs, filenames = IO.load_annot_tiffs(data_path, ext='ome.tiff')

# TODO: rerun on full z-slices after solving the misalignment issues
cyif_imgs, filenames = cyif_imgs[:7], filenames[:7]
filenames

In [ ]:
chan_labels = cyif_imgs[0].keys()
print(list(chan_labels))

#### Examine continuity of zonation markers:

In [ ]:
# Compute ROI (foreground) mask  + AF mask for each z-slice
# roi_masks = [get_roi_mask(img['DAPI'])
#              for img in cyif_imgs]

af1_masks = [(img['Sample AF_01'] > 40).astype(np.uint8)
             for img in cyif_imgs]
af2_masks = [(img['Sample AF_02'] > 40).astype(np.uint8)
             for img in cyif_imgs]

In [ ]:
# Try subtracting w/ AF within ROI foregrond regions
# Normalize each channel to [0, 1]

zone_labels = ['GS 647', 'CYP3A4', 'ASS1 PE', 'Col I', 'Pan CK']
zonation_chans = {}

for label in zone_labels:
    chans = []
    for i, (img, roi) in enumerate(zip(cyif_imgs, roi_masks)):
        af = af1_masks[i] if label in zone_labels[:3] else af2_masks[i]
        
        chan = rescale_intensity(img[label], out_range=(0, 1)) - af
        chan[chan < 0] = 0
        chan[roi == 0] = 0
        chans.append(chan)

        # chan = rescale_intensity(img[label].copy(), out_range=(0, 1))
        # chan[roi == 0] = 0
        # chans.append(chan)
        
    zonation_chans[label] = np.array(chans)

del label, chans, img, roi, chan, af
# del img, roi, chan

In [ ]:
def disp_chans(img, title=None, ncols=4):
    depth = len(img)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows))
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(img[idx])
            idx += 1
            
    plt.tight_layout()
    plt.suptitle(title, y=1.01)
    plt.show()
    return None

In [ ]:
# Denoise w/ binarized AF
for label in zone_labels:
    disp_chans(zonation_chans[label], title=label)
del label

Signals of `GS`, `CYP` & partial `ASS` looks Okay. `Collagen` & `Pan-CK` seems X usable. Quantify slice-wise transitions:

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 3))
ax1.imshow(zonation_chans['GS 647'].max(0))
ax1.set_title('GS')
ax2.imshow(zonation_chans['CYP3A4'].max(0))
ax2.set_title('CYP')
ax3.imshow(zonation_chans['ASS1 PE'].max(0))
ax3.set_title('ASS')

plt.suptitle("Maximum Intensity Projection on Z-slices", y=1.01)
plt.tight_layout()
plt.show()

**Proof-of-concept** of cross-slice registration quality

In [ ]:
# Save sample overlayed zonation markers
for label, chan in zonation_chans.items():
    tifffile.imwrite('../data/cycif/backup/CyIF_{}_slices.tif'.format(label), chan, metadata={'axes': 'CYX'})

In [ ]:
def disp_optical_flow(z1, z2, title=None):
    # Reference: 
    # https://scikit-image.org/docs/stable/auto_examples/registration/plot_opticalflow.html
    v, u = optical_flow_ilk(z1, z2)
    norm = np.sqrt(u** + v**2)
    nvec = 20
    nl, nc = z1.shape
    step = max(nl//nvec, nc//nvec)
    
    y, x = np.mgrid[:nl:step, :nc:step]
    u_ = u[::step, ::step]
    v_ = v[::step, ::step]

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 3))
    ax1.imshow(z1)
    ax1.set_title('Slice z')

    ax2.imshow(z2)
    ax2.set_title('Slice z+1')
    
    ax3.imshow(norm)
    ax3.quiver(x, y, u_, v_, color='r', units='dots',
               angles='xy', scale=0.8, scale_units='xy', lw=2)
    ax3.set_title("Optical flow vector field")

    plt.tight_layout()
    plt.suptitle(title, y=1.1)
    plt.show()

In [ ]:
for label in zone_labels[:3]:
    print('Channel', label)
    print('==='*5)
    for i in range(len(zonation_chans[label])-1):
        z1, z2 = zonation_chans[label][i], zonation_chans[label][i+1]
        disp_optical_flow(z1, z2, title='Transition of {0} slices({1} vs. {2})'.format(label, i, i+1))   
    print('\n\n\n')

del z1, z2

#### 3D heat diffusion 
- Trajectory origin: `ASS` for now; destination `GS`
- 6-connected components as the graph unit

In [ ]:
import networkx as nx
from scipy.sparse import linalg
from skimage.segmentation import find_boundaries
from skimage.morphology import binary_dilation


In [ ]:
# TODO: add anisotropy scale along Z-axis in the future
def create_graph(roi):
    """
    Convert combinatory graph w/ 4-connected compoments:
    adj(x) = {(i+1, j, k), (i-1, j, k),
              (i, j+1, k), (i, j-1, k),
              (i, j, k+1), (i, j, k-1)}
    from corresponding ROI pixels from the input image
    
    Code Referenced from: https://stackoverflow.com/questions/63653267/how-to-create-a-graph-with-an-images-pixel
    """
    # Horizontal edge (X)
    hx, hy, hz =  np.nonzero(np.logical_and(roi[1:] == 1, roi[:-1] == 1))  # horizontal edge start positions
    h_units = np.array([hx, hy, hz]).T
    h_starts = [tuple(n) for n in h_units]
    h_ends = [tuple(n) for n in h_units + (1, 0, 0)] 
    horizontal_edges = zip(h_starts, h_ends)

    # Vertical edge (Y)
    vx, vy, vz = np.nonzero(np.logical_and(roi[:,1:,:] == 1, roi[:,:-1,:] == 1))  # vertical edge start positions
    v_units = np.array([vx, vy, vz]).T
    v_starts = [tuple(n) for n in v_units]
    v_ends = [tuple(n) for n in v_units + (0, 1, 0)] 
    vertical_edges = zip(v_starts, v_ends)

    # In-plane edge (Z)
    ux, uy, uz = np.nonzero(np.logical_and(roi[:,:,1:] == 1, roi[:,:,:-1] == 1))
    u_units = np.array([ux, uy, uz]).T
    u_starts = [tuple(n) for n in u_units]
    u_ends = [tuple(n) for n in u_units + (0, 0, 1)]
    inplane_edges = zip(u_starts, u_ends)

    # Create graph
    G = nx.Graph()
    G.add_edges_from(horizontal_edges)
    G.add_edges_from(vertical_edges)
    G.add_edges_from(inplane_edges)
    
    return G


def add_graph_props(G, cv_nodes, pv_nodes, depth):
    """
    Assign init. temp. & ROI boundary as graph properties
    """
    for n in G:
        if n in cv_nodes:
            G.nodes[n]['t'] = 1
            G.nodes[n]['bound'] = True
        elif n in pv_nodes:
            G.nodes[n]['t'] = -1
            G.nodes[n]['bound'] = True
        else:
            G.nodes[n]['t'] = 0
            # Only define ROI bound along XY plane
            if G.degree[n] < 6 and 0 < n[0] < depth-1: # 3D
                G.nodes[n]['bound'] = True
    return None


def compute_interior_temp(G, debug=False):
    """
    Compute temperature of "interior" nodes based on Harmonic interpolation solution (Grady & Schwartz, 2003)
    """
    # Constructed permuted Laplacian Matrix L => {L_b, L_i, R, R^T}
    bound_nodes = [
        n for n, v in G.nodes.items()
        if 'bound' in v
    ]

    interior_nodes = [
        n for n, v in G.nodes.items()
        if 'bound' not in v
    ]

    n_bound = len(bound_nodes)
    perm_node_orders = bound_nodes + interior_nodes
    if debug:
        assert len(G) == len(perm_node_orders)

    L = nx.laplacian_matrix(G, nodelist=perm_node_orders)
    L_i = L[n_bound:, n_bound:]
    R = L[:n_bound, n_bound:]

    # Validate permuted nodes' in-degree have the correct order [d(bound), d(interior)]
    if debug:
        diag = np.diag(L)
        for i, n in enumerate(perm_node_orders):
            assert G.degree[n] == diag[i]

    # Compute interior temperature u(i) from L & u(b): 
    # Sol:  L_i @ u(i) = -R^T @ u(b)
    u_b = np.asarray([G.nodes[n]['t'] for n in bound_nodes])
    u_i = linalg.cg(A=L_i, b=-R.T@u_b) 

    if isinstance(u_i, tuple):
        u_i = u_i[0]
    
    return u_i, tuple(np.array(interior_nodes).T)  # 2 x N tuple


def assign_diffusion_temp(
    u_i, 
    interior_coords,
    cv_coords,
    pv_coords, 
    shape
):
    """
    Assign steady-state sol. of the diffused pixel values back to the image
    """
    assert len(interior_coords[0]) == len(u_i), 'Different coords & temperature lengths'
    u = np.zeros(shape, dtype=np.float64)
    u[interior_coords] = u_i
    u[cv_coords] = 1
    u[pv_coords] = -1
    return u


def discretize_temp(u, roi, cv_coords, n_layers=10, return_border=False, verbose=False):
    """
    Create discretized 1-indexed bins (1,2,...,n) as the zonation priors 
    from diffused gradient temperature `u`, keep CV & PV regions off from 
    `roi` as the min (PV) / max (CV) zones
    """
    assert n_layers > 3, "Invalid `n_layers`, please assign # lobule layers > 3"
    
    coords = np.nonzero(roi)
    coords_to_rm = np.nonzero(1-roi)
    qs = np.quantile(u[coords], np.linspace(0, 1, n_layers-1))

    if verbose:
        print('Quantile:', qs)
        
    zone = np.zeros_like(u, dtype=np.int32)
    for i, q in enumerate(qs[:-1]):
        zone[u >= q] = i+1
    zone[coords_to_rm] = 0

    zone[cv_coords] = zone.max() + 1
    zone += 1

    # Assign 1-pixel width border btw zones
    if return_border:
        border = find_boundaries(zone)
        zone[border] = 0

    return zone

**(1)**. Construct 3D ROI & CV / PV tentative mask

**TODO**: expand # Z-slices & increase weights along in-plane (Z-axis) edges for anisotropy

In [ ]:
# CV / PV mask w/ otsu threshold

# roi = np.array(roi_masks)
# cv_mask = np.zeros_like(roi, dtype=np.int32)
# for i, chan in enumerate(zonation_chans['GS 647']):
#     threshold = threshold_otsu(chan)
#     cv_mask[i] = (chan > threshold).astype(np.int32)
# del chan, threshold

# pv_mask = np.zeros_like(roi, dtype=np.int32)
# for i, chan in enumerate(zonation_chans['ASS1 PE']):
#     threshold = threshold_otsu(chan)
#     pv_mask[i] = (chan > threshold).astype(np.int32)
# del chan, threshold

# mask = np.zeros_like(roi, dtype=np.int32)
# mask[np.logical_and(pv_mask == 1, cv_mask == 0)] = -1
# mask[np.logical_and(pv_mask == 0, cv_mask == 1)] = 1

# Dilate on mask
# for i, mask_slc in enumerate(mask):
#     cv_mask = binary_dilation(mask_slc == 1, np.ones((5, 5))).astype(np.uint8)
#     pv_mask = binary_dilation(mask_slc == -1, np.ones((5, 5))).astype(np.uint8)

#     mask[i][pv_mask == 1] = -1
#     mask[i][cv_mask == 1] = 1

# del cv_mask, pv_mask

# Load computed roi & mask
# DEBUG: 3D diffusion's correction w/ Z-slices 8-11
roi = tifffile.imread('../data/cycif/backup/CyIF_ROI.tif')[-4:]
mask = tifffile.imread('../data/cycif/backup/CyIF_zone_masks.tif')[-4:]

In [ ]:
for i, (roi_slc, mask_slc) in enumerate(zip(roi, mask)):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    ax1.imshow(roi_slc)
    ax1.set_title('Tissue ROI (z={})'.format(i))
    ax1.axis('off')
    
    ax2.imshow(mask_slc, cmap='coolwarm')
    ax2.set_title('CV/PV mask (z={})'.format(i))
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()

del roi_slc, mask_slc

In [ ]:
# Save current ROI & CV/PV mask
tifffile.imwrite('../data/cycif/backup/CyIF_ROI.tif', roi)
tifffile.imwrite('../data/cycif/backup/CyIF_zone_masks.tif', mask)

In [ ]:
cv_indices = {tuple(idx) for idx in np.array(np.where(mask == 1)).T}
pv_indices = {tuple(idx) for idx in np.array(np.where(mask == -1)).T}

**(2)** Construct 3D graph & run diffusion inference

In [ ]:
# DEBUG 3D graph construction
G = create_graph(roi=roi)
add_graph_props(G, cv_indices, pv_indices, depth=roi.shape[0])
u_i, interior_nodes = compute_interior_temp(G)
u = assign_diffusion_temp(u_i, 
                          interior_nodes,
                          cv_coords=np.where(mask == 1),
                          pv_coords=np.where(mask == -1),
                          shape=mask.shape)

In [ ]:
for i, u_slc in enumerate(u):
    plt.figure()
    plt.imshow(u_slc, cmap='coolwarm')
    plt.title('Diffused dynamics (z={})'.format(i))
    plt.show()

In [ ]:
# Save inferred heat dynamics
tifffile.imwrite('../data/cycif/backup/CyIF_zone_dynamics.ome.tif', u, metadata={'axes': 'ZYX'})

### Liver Zonation Classification


- Zone 1: (optional): Collagen (PV / PT)
- Zone 2: `ASS` (Peri-portal traid): 
- Zone 2': `CYP` (Peri-central vein)
- Zone 3: `GS` (CV)


In [ ]:
import time
import pickle

from skimage.exposure import rescale_intensity, equalize_adapthist
from skimage.filters import gaussian, sobel, threshold_otsu

from scipy import ndimage as ndi
from cellpose.transforms import normalize99

In [ ]:
# Helper functions
# Simple morphological processing to remove holes & FP fluorescence islands
def remove_holes(roi, min_area=20000):
    """
    Remove holes & FP lslands in binary ROI mask
    """
    roi_filtered = roi.copy().astype(np.uint8)
    roi_labeled, n_features = ndi.label(roi)
    
    for i in range(1, n_features+1):
        if (roi_labeled == i).sum() < min_area:
            roi_filtered[roi_labeled == i] = 0
            
    return ndi.binary_fill_holes(roi_filtered)


In [ ]:
# Compute expression distribution along PV - CV trajectory
def calc_moving_avg(x, window_size=100):
    assert window_size > 1 and window_size < len(x), "Window-size should be smaller than array length"    
    return np.convolve(x, np.ones(window_size)/window_size, mode='valid')


def get_colors(inp, colormap, vmin=None, vmax=None):
    norm = plt.Normalize(vmin, vmax)
    return colormap(norm(inp))


def disp_trajectory_exprs(
    diff_mask, 
    exprs,
    roi,
    labels=None,
    index_step=1000,
    window_size=100,
    figsize=(10, 5)
):
    if isinstance(labels, list):
        assert len(labels) == len(exprs)

    # Subsample indice w/ ascending vals
    fg_coords = np.nonzero(roi)
    gradients = diff_mask[fg_coords]
    size = len(gradients)
    ss_idx = np.argsort(gradients)[0:size:index_step]  
    
    # Calculate moving avg. expressions per marker
    smoothed_exprs = []
    for expr in exprs:
        expr_1d = expr[fg_coords][ss_idx]
        smoothed_exprs.append(calc_moving_avg(expr_1d, window_size=window_size))
        
    xx = np.linspace(0, 1, len(smoothed_exprs[0]))
    
    fig, ax = plt.subplots(figsize=figsize, dpi=200)
    for lbl, expr in zip(labels, smoothed_exprs):
        ax.plot(xx, expr, '-', linewidth=1, label=lbl)
        
    ax.legend()
    ax.set_xlabel('PV --> CV')
    ax.set_ylabel('Smoothed intensity')
    plt.show()
    
    return None


In [ ]:
# Compute expression distribution in each lobule layer
def draw_brace(ax, xspan, text):
    """
    Draws an annotated brace on the axes
    
    Reference: 
    https://stackoverflow.com/questions/18386210/annotating-ranges-of-data
    """
    
    xmin, xmax = xspan
    xspan = xmax - xmin
    ax_xmin, ax_xmax = ax.get_xlim()
    xax_span = ax_xmax - ax_xmin
    ymin, ymax = ax.get_ylim()
    yspan = ymax - ymin
    resolution = int(xspan/xax_span*100)*2+1 # guaranteed uneven
    beta = 300./xax_span # the higher this is, the smaller the radius

    x = np.linspace(xmin, xmax, resolution)
    x_half = x[:resolution//2+1]
    y_half_brace = (1/(1.+np.exp(-beta*(x_half-x_half[0])))
                    + 1/(1.+np.exp(-beta*(x_half-x_half[-1]))))
    y = np.concatenate((y_half_brace, y_half_brace[-2::-1]))
    y = ymin + (.05*y - .01)*yspan # adjust vertical position

    ax.autoscale(False)
    ax.plot(x, y, color='black', lw=1)

    ax.text((xmax+xmin)/2., ymin+.07*yspan, text, fontsize=5, c='blue', ha='center', va='bottom')
    
    
def disp_layer_exprs(
    layer_mask, 
    expr, 
    label=None,
    n_samples=5000,
    figsize=(6, 3)
):
    assert layer_mask.shape == expr.shape, "Inconsistent shapes btw lobule mask / expr matrix / roi mask"

    nlayers = layer_mask.max()
    bs_exprs = np.zeros((2, nlayers*n_samples))  # Bootstrap indices in each lobule layer
    
    for i in range(nlayers):
        coords = np.array(np.where(layer_mask == i+1)) # 1-index lobule layer     
        ss_idx = np.random.randint(0, coords.shape[1], n_samples)
        ss_coords = tuple(coords[:, ss_idx])

        bs_exprs[0, i*n_samples:(i+1)*n_samples] = i+1
        bs_exprs[1, i*n_samples:(i+1)*n_samples] = expr[ss_coords]
        
    bs_expr_df = pd.DataFrame(bs_exprs.T, columns=['Layer', 'Expression'])
    bs_expr_df.iloc[:, 0].astype(np.int32)
    
    fig, ax = plt.subplots(figsize=figsize, dpi=200)
    sns.violinplot(x='Layer', y='Expression', data=bs_expr_df, ax=ax, scale='width')
    ax.set_title(label)
    
    # Annotate lobule layer zonations
    draw_brace(ax, (0, 1), 'PV')
    draw_brace(ax, (1, 3), 'Peri-portal')
    draw_brace(ax, (3, 6), 'Intermediate')
    draw_brace(ax, (6, 8), 'Peri-central')
    draw_brace(ax, (8, 9), 'CV')
    
    plt.show()
    
    return None

#### Presha's slide

In [ ]:
overlaid_img = tifffile.imread('data/cycif/pilot_fov/Layer_overlaid_ROI.ome.tif')
print(overlaid_img.shape)

Define the image boundary & ROI (smoothed DAPI):

In [ ]:
smoothed_dapi = rescale_intensity(gaussian(overlaid_img[0], sigma=10), out_range=(0, 1))
roi_thld = threshold_otsu(smoothed_dapi)

# plt.hist(smoothed_dapi.flatten(), bins=30)
# plt.axvline(x=roi_thld, c='r')
# plt.show()

roi = remove_holes(smoothed_dapi > roi_thld)
roi[:200, :] = 0 # Heuristically remove fluorescence artifacts
plt.imshow(roi)
plt.show()

Compute zonation gradient from CV & PV mask

In [ ]:
cv_pv_mask = tifffile.imread('data/cycif/pilot_fov/Overlaid_ROI_zone_masks.ome.tif')
cv_pv_mask[roi == 0] = 0
plt.figure()
plt.imshow(cv_pv_mask)
plt.show()


In [ ]:
# Mask out background region
overlaid_img[:, roi == 0] = 0

# Parse CV & PV annotations
pv_mask = (cv_pv_mask == 1).astype(np.uint8)
cv_mask = (cv_pv_mask == 2).astype(np.uint8)

In [ ]:
labels = ['GS', 'CYP', 'ASS', 'CD56', 'CD31']
chans = overlaid_img[[5, 2, 1, 3, 6], :]  # GS, CYP, ASS, CD56, Pan-CK, CD31
zone_marker_channels = [
    rescale_intensity(
        gaussian(
            normalize99(chans[i], lower=10, upper=95),
            sigma=1, preserve_range=True
        ),
        out_range=(0, 1)
    )
    for i in range(len(chans))
]
gs_chan, cyp_chan, ass_chan, cd56_chan, cd31_chan = zone_marker_channels
del chans

In [ ]:
t0 = time.perf_counter()
diff_mask = iter_diffusion(
    cv_mask,
    pv_mask,
    roi=roi,
    init_val=1e-3,
    step=10,
    max_iter=30,
    verbose=True
)
t1 = time.perf_counter()

print('Heat diffusion took {} seconds'.format(t1-t0))
plt.imshow(diff_mask, cmap='coolwarm')
plt.show()

In [ ]:
# Gradient
plt.figure(dpi=200)
plt.imshow(diff_mask, cmap='coolwarm')
plt.suptitle('PV-CV gradient')
plt.axis('off')
plt.show()

In [ ]:
# PC-PP boundary
pc_pp_boundary = np.isclose(diff_mask, 0, atol=1e-5)
pc_pp_boundary[roi == 0] = 0

plt.figure(dpi=200)
plt.imshow(pc_pp_boundary)
plt.axis('off')
plt.suptitle('PP-PC boundary')
plt.show()

In [ ]:
lobule_layer_mask = bin_channel(
    diff_mask, 
    roi=roi,
    qrange=np.linspace(0, 1, 11),
    verbose=True
).astype(np.int64)

In [ ]:
plt.figure(dpi=200)
im = plt.imshow(lobule_layer_mask)
plt.colorbar(im)

plt.axis('off')
plt.suptitle('Lobule layers along PV-CV gradient')
plt.show()

In [ ]:
b# Plot zone marker expressions vs. lobule gradients
disp_trajectory_exprs(
    diff_mask,
    exprs=[gs_chan, cyp_chan, ass_chan],
    roi=roi,
    labels=['GS', 'CYP', 'ASS'],
    window_size=300,
    figsize=(10, 5)
)

In [ ]:
# Plot zone marker expressions vs. lobule gradients
disp_trajectory_exprs(
    diff_mask,
    exprs=[cd56_chan, cd31_chan],
    roi=roi,
    labels=['CD56', 'CD31'],
    window_size=300,
    figsize=(10, 5)
)

In [ ]:
# Plot zone marker expressions vs. lobule layers
disp_layer_exprs(
    lobule_layer_mask, 
    gs_chan, 
    label='GS',
    n_samples=500,
)

disp_layer_exprs(
    lobule_layer_mask, 
    cyp_chan, 
    label='CYP',
    n_samples=500,
)

disp_layer_exprs(
    lobule_layer_mask, 
    ass_chan, 
    label='ASS',
    n_samples=500,
)

In [ ]:
# Plot other immune marker expr. distributiosn vs. lobule layers
disp_layer_exprs(
    lobule_layer_mask, 
    cd56_chan, 
    label='CD56',
    n_samples=500,
)

disp_layer_exprs(
    lobule_layer_mask, 
    cd31_chan, 
    label='CD31',
    n_samples=500,
)

#### Aubrianna's sample slide patch (w/ Collagen marker)

Channels: 

| Cycle | 0 | 1 | 2 | 3 |
| --- | --- | --- | --- | --- |
| 0 |  LYVE1 | CD68 | CD45 | DAPI | 
| 1 |  **GS** | CD3E | **PanCK** | - |
| 2 |  CD163 | HNF4a | CD56 | - |
| 3 |  **CYP3A4** | Vimentin | **Col1A1** | - |
| 4 |  **GS** | **ASS1** | aSMA | - |

In [ ]:
with open('data/cycif/pilot_fov/mini-ibex-img-list.pkl', 'rb') as ifile:
    cycif_arr = pickle.load(ifile)

##### (1). Dinstinct zonation segmentation

In [ ]:
# save to ome.tif file

zone_labels = ['GS', 'CYP', 'ASS', 'Pan-CK']

chans = np.array([
    cycif_arr[1][0],  # GS
    cycif_arr[3][0],  # CYP
    cycif_arr[4][1],  # ASS
    cycif_arr[1][2],  # Pan-CK
]).copy()

# Normalize intensities
# Filter out bottom 20% & top 5% intensities
zone_marker_channels = [
    rescale_intensity(
        gaussian(
            normalize99(chans[i], lower=10, upper=95),
            sigma=100, preserve_range=True
        ),
        out_range=(0, 1)
    )
    for i in range(len(chans))
]

del chans
# tifffile.imwrite('data/cycif/mini_ibex.ome.tif', cycif_normed_channels, metadata={'axes': 'CYX'})

In [ ]:
hepa_labels = [
    'LYVE1',
    'CD68',
    'CD45',
    'CD3E',
    'CD163',
    'CD56',
]

chans = np.array([
    cycif_arr[0][0], # LYVE1
    cycif_arr[0][1], # CD68
    cycif_arr[0][2], # CD45
    cycif_arr[1][1], # CD3E
    cycif_arr[2][0], # CD163
    cycif_arr[2][2], # CD56
])

hepa_channels = [
    rescale_intensity(
        gaussian(
            normalize99(chans[i], lower=10, upper=95),
            sigma=5, preserve_range=True
        ),
        out_range=(0, 1)
    )
    for i in range(len(hepa_labels))
]

In [ ]:
# tifffile.imwrite('data/cycif/mini_ibex.ome.tif', cycif_normed_channels, metadata={'axes':'CYX'})
tifffile.imwrite('data/cycif/pilot_fov/mini_ibex.hepa_channels.tif', chans, metadata={'axes': 'CYX'})

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for (channel, ax, img) in zip(zone_labels, axes, zone_marker_channels):
    ax.imshow(img, cmap='coolwarm')
    ax.axis('off')
    ax.set_title(channel)
    
plt.tight_layout()
plt.show()
                
del channel, ax, img

Define binary masks for CV & PV: 

- CV: `GS+`
- PV: `Pan-CK+ & GS-`

In [ ]:
gs_chan = zone_marker_channels[0].copy()
cyp_chan = zone_marker_channels[1].copy()
ass_chan = zone_marker_channels[2].copy()
panck_chan = zone_marker_channels[3].copy()


In [ ]:
# Binarize CV & PV regions
roi = np.ones_like(gs_chan).astype(np.uint8)
qrange = np.linspace(0, 1, 15)

gs_quantized = bin_channel(gs_chan, roi=roi, qrange=qrange)
panck_quantized = bin_channel(panck_chan, roi=roi, qrange=qrange)

In [ ]:
# Filter out FP PV masks by ASS expressions
def filter_pv_mask(candidate_mask, ass_expr, thld=0.1):
    """
    Filter false-positive PV mask without ASS (peri-portal marker) colocalization
    """
    filtered_mask = candidate_mask.copy()
    mask_lbls, nlbls = ndi.label(candidate_mask)
    for i in range(nlbls):
        if ass_expr[mask_lbls == i+1].mean() < thld:
            filtered_mask[mask_lbls == i+1] = 0

    return filtered_mask

In [ ]:
# Criteria: Pan-CK+, dilation(GS)- <==> co-localize with periportal marker (ASS)
pv_mask = filter_pv_mask(panck_mask, ass_chan)

plt.figure()
plt.imshow(pv_mask)
plt.suptitle('PV mask')
plt.show()

In [ ]:
# Save CV & PV mask
zone_mask = pv_mask + cv_mask*2
tifffile.imsave(os.path.join('data/cycif/pilot_fov/mini-ibex-img_zonation.tif'), zone_mask)

##### (2). Gradient distance definition

Try iterative dilation to assign CV - PV gradient:

In [ ]:
# Load CV-PV mask
zone_mask = tifffile.imread('data/cycif/pilot_fov/mini-ibex-img_zonation.tif')
pv_mask = (zone_mask == 1).astype(np.uint8)
cv_mask = (zone_mask == 2).astype(np.uint8)


In [ ]:
t0 = time.perf_counter()
diff_mask = iter_diffusion(
    cv_mask,
    pv_mask,
    roi=np.ones_like(cv_mask).astype(np.uint8),
    init_val=1e-3,
    step=100,
    max_iter=30,
    verbose=True
)
t1 = time.perf_counter()

print('Heat diffusion took {} seconds'.format(t1-t0))
plt.imshow(diff_mask, cmap='coolwarm')
plt.show()

In [ ]:
tifffile.imsave('data/cycif/pilot_fov/mini-ibex-img_lobule_grad.tif', diff_mask)

In [ ]:
# Plot zone marker expr. distributions vs. lobule gradients
disp_trajectory_exprs(
    diff_mask,
    exprs=[gs_chan, cyp_chan, ass_chan, panck_chan],
    zonation_mask=zone_mask,
    labels=['GS', 'CYP', 'ASS', 'Pan-CK'],
    window_size=300,
    figsize=(10, 5)
)

In [ ]:
# Plot other marker expr. distributions vs. lobule gradients
disp_trajectory_exprs(
    diff_mask,
    exprs=hepa_channels,
    zonation_mask=zone_mask,
    labels=hepa_labels,
    window_size=300,
    figsize=(10, 5)
)

In [ ]:
# Plot other immune / hepa. markers
for expr, lbl in zip(hepa_channels, hepa_labels):
    disp_trajectory_exprs(
        diff_mask,
        exprs=[expr],
        zonation_mask=zone_mask,
        labels=[lbl],
        window_size=300,
        figsize=(6, 3)
    )
               
del expr

**Quantize lobule sub-zonations along PV-CV lobular axis**

CV + PV + 8 lobule layers btw PP - PC zones

In [ ]:
diff_mask = tifffile.imread('data/cycif/pilot_fov/mini-ibex-img_lobule.tif')
cv_pv_mask = tifffile.imread('data/cycif/pilot_fov/mini-ibex-img_zonation.tif')

In [ ]:
np.linspace(0, 1, 9)

In [ ]:
lobule_layer_mask = bin_channel(
    diff_mask, 
    #roi=np.ones_like(diff_mask).astype(np.uint8),
    roi=(cv_pv_mask == 0).astype(np.uint8),
    #qrange=np.linspace(0, 1, 11),
    qrange=np.linspace(0, 1, 9),
).astype(np.int64)

lobule_layer_mask[cv_pv_mask == 2] = lobule_layer_mask.max()+1
lobule_layer_mask += 1

# Assign 1-index-width boundary between each layer 
from skimage.segmentation import find_boundaries
boundary = find_boundaries(lobule_layer_mask)
lobule_layer_mask[boundary] = 0

In [ ]:
# PC-PP boundary 
plt.figure()
plt.imshow(np.isclose(diff_mask, 0, atol=1e-5))
plt.show()

In [ ]:
plt.figure(dpi=200)
im = plt.imshow(lobule_layer_mask)
plt.colorbar(im)
plt.suptitle('Lobule layers along PV-CV gradient')
plt.axis('off')
plt.show()

del im


In [ ]:
tifffile.imsave('data/cycif/pilot_fov/mini-ibex-img_lobule_layers.tif', lobule_layer_mask)

In [ ]:
# Plot zonation marker expr. distribution vs. lobule layers
disp_layer_exprs(lobule_layer_mask, gs_chan, label='GS')
disp_layer_exprs(lobule_layer_mask, cyp_chan, label='CYP')
disp_layer_exprs(lobule_layer_mask, ass_chan, label='ASS')
disp_layer_exprs(lobule_layer_mask, panck_chan, label='Pan-CK')


In [ ]:
# Plot other immune & hepatocyte markers vs. lobule layers
for expr, lbl in zip(hepa_channels, hepa_labels):
    disp_layer_exprs(
        lobule_layer_mask,
        expr=expr,
        label=lbl
    )
    
del expr, lbl

In [ ]:
# Zonation expression heatmaps
def disp_layer_exprs(
    layer_mask, 
    exprs, 
    labels=None,
    n_samples=5000,
    figsize=(6, 3)
):
    layer_indices = np.arange(0, layer_mask.max()+1)
    bs_exprs = np.zeros((2, len(layer_indices)*n_samples))  # Bootstrap indices in each lobule layer
    
    for idx in layer_indices:
        coords = np.array(np.where(layer_mask == idx))     
        ss_idx = np.random.randint(0, coords.shape[1], n_samples)
        ss_coords = tuple(coords[:, ss_idx])
        
        bs_exprs[0, idx*n_samples:(idx+1)*n_samples] = idx
        bs_exprs[1, idx*n_samples:(idx+1)*n_samples] = expr[ss_coords]
        
    bs_expr_df = pd.DataFrame(bs_exprs.T, columns=['Layer', 'Expression'])
    bs_expr_df.iloc[:, 0].astype(np.int32)
    
    fig, ax = plt.subplots(figsize=figsize, dpi=200)
    sns.heatmap(x='Layer', y='Expression', data=bs_expr_df, ax=ax)
    ax.set_title(label)
    plt.show()
    
    return None

Classify zonations into lobuli regions (trying Delauney triangulation / Voronoi diagram)

In [ ]:
cv_labeled_mask, cv_mass_centers = get_mass_centers(cv_mask)
pv_labeled_mask, pv_mass_centers = get_mass_centers(pv_mask)
cv_nn_indices = get_nn_assignment(cv_mass_centers, pv_mass_centers) # CV indices closest to PV index 1, 2, etc.

cv_pv_mask = cv_mask + pv_mask
combined_centers = np.vstack([cv_mass_centers, pv_mass_centers])

In [ ]:
cv_pv_mask = cv_mask + pv_mask
combined_centers = np.vstack([cv_mass_centers, pv_mass_centers])

In [ ]:
def get_voronoi(seeds, shape):
    """
    Code reference:
    https://learnopencv.com/delaunay-triangulation-and-voronoi-diagram-using-opencv-c-python/
    """
    
    rect = (0, 0, shape[1], shape[0])
    subdiv = cv2.Subdiv2D(rect)
    
    # Compute Voronoi tessellation
    for p in seeds:
        subdiv.insert(tuple([p[1], p[0]]))
    
    (facets, centers) = subdiv.getVoronoiFacetList([])
    mask = np.zeros(shape, dtype=np.int32)
    
    for i, facet in enumerate(facets):
        ifacet_arr = []
        for f in facet:
            ifacet_arr.append(f)
            
        # Fill region
        ifacet = np.array(ifacet_arr, np.int32)
        # color = (np.random.randint(0, 255), np.random.randint(0, 255), np.random.randint(0, 255))
        cv2.fillConvexPoly(mask, ifacet, i+1, cv2.LINE_AA, 0)
        
        # Fill ridge
        ifacets = np.array([ifacet])
        cv2.polylines(mask, ifacets, True, (0, 0, 0), 2, cv2.LINE_AA, 0)
        
    return mask
    

In [ ]:
vor_mask = get_voronoi(combined_centers, cv_mask.shape)
vor_boundary = (vor_mask == 0).astype(np.uint8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), dpi=200)
ax1.imshow(vor_mask)
ax1.axis('off')

ax2.imshow(cv_mask + pv_mask*2 + vor_boundary*3)
ax2.axis('off')

plt.tight_layout()
plt.show()

##### (3). Graph-based heat diffusion 

Aim: implement graph-based (a. 8-neighbor model for pixel-level; b. delaunay tesselation graph for cell-level) heat diffusion w/ analytical sol. from Harmonic function w/ boundary conditions:

1. Fixed heat source (+1) & sink (-1)
2. Zero for boundary Pixel / Node


- CD68: Pan-Macrophage
- LYVE1: Sinusoidal Epithelial
- Vimentin: Mesenchymal

In [ ]:
from skimage import io

In [ ]:
# Load segmentation results
vimentin_dat = np.load('data/cycif/pilot_fov/vimentin_seg.npy', allow_pickle=True).item()
vimentin_mask = vimentin_dat['masks']

cd68_mask = io.imread('data/cycif/pilot_fov/cd68_masks.png')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(cp_plot.mask_rgb(vimentin_mask))
ax1.set_title('Vimentin (Stromal)')
ax1.axis('off')

ax2.imshow(cp_plot.mask_rgb(cd68_mask))
ax2.set_title('CD68 (Macrophage)')
ax2.axis('off')
plt.show()

In [ ]:
n_overlap_masks = ndi.label(np.logical_and(vimentin_mask > 0, cd68_mask > 0))[1]
print(
    ' # Stromal masks', len(np.unique(vimentin_mask)[1:]), '\n',
    '# Macrophage masks', len(np.unique(cd68_mask)[1:]), '\n',
    '# Overlap masks', n_overlap_masks
)

---